###### Import Libraries

In [3]:
import pandas as pd
import numpy as np

###### Load data

In [4]:
df = pd.read_csv(r"H:\Downloads\marketing_campaign_data_messy.csv")
print(f"Loaded Dataset: {df.shape[0]} rows, {df.shape[1]} columns")

Loaded Dataset: 2020 rows, 12 columns


### 1. Header cleaning

In [6]:
print("Before:")
print(df.columns.tolist())

Before:
[' Campaign_ID ', 'Campaign_Name', 'Start_Date', 'End_Date', 'Channel', 'Impressions', 'Clicks ', 'Spend', 'Conversions', 'Active', 'Clicks', 'Campaign_Tag']


In [16]:
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

print("Fix applied! After:")
print(df.columns.tolist())

Fix applied! After:
['campaign_id', 'campaign_name', 'start_date', 'end_date', 'channel', 'impressions', 'clicks', 'spend', 'conversions', 'active', 'clicks', 'campaign_tag']


### 2. Type Conversion and Currency Cleaning

In [17]:
print("Before:")
spend_column = df['spend'].astype(str).str.contains(r'\$')
print(df.loc[spend_column, ['campaign_id', 'spend']].head(5))

Before:
   campaign_id     spend
0    CMP-00001   $102.82
21   CMP-00022   $2428.4
22   CMP-00023  $4726.22
31   CMP-00032  $2759.35
32   CMP-00033  $2393.02


In [18]:
df['spend'] = df['spend'].astype(str).str.replace(r'[^\d.-]', '', regex=True)
df['spend'] = pd.to_numeric(df['spend'])

print("Fix applied! After:")

print(df.loc[spend_column, ['campaign_id', 'spend']].head(5))

Fix applied! After:
   campaign_id    spend
0    CMP-00001   102.82
21   CMP-00022  2428.40
22   CMP-00023  4726.22
31   CMP-00032  2759.35
32   CMP-00033  2393.02


### 3. Categorical Typos

In [19]:
print("Before: ")
print(df['channel'].unique())

Before: 
['TikTok' 'Facebook' 'Email' 'Instagram' 'Google Ads' 'E-mail' nan 'Gogle'
 'Tik_Tok' 'Facebok' 'Insta_gram']


In [20]:
cleanup_column = {
    'Facebok': 'Facebook',
    'Insta_gram': 'Instagram',
    'Gogle': 'Google Ads',
    'Tik_Tok': 'TikTok',
    'E-mail': 'Email',
    'N/A': np.nan # Handling the ghost value here too
}

df['channel'] = df['channel'].replace(cleanup_column)

print("Fix applied! After:")
print(df['channel'].unique())

Fix applied! After:
['TikTok' 'Facebook' 'Email' 'Instagram' 'Google Ads' nan]


### 4. Handling Mixed Booleans

In [21]:
print("Before: ")
print(df['active'].unique())

Before: 
['Y' '0' 'No' 'True' 'Yes' '1' 'False']


In [23]:
bool_column = {
    'Yes': True,
    'Y': True,
    '1': True,
    1: True,
    'No': False,
    '0': False,
    0: False
}

df['active'] = df['active'].map(bool_column).fillna(False).astype(bool)

print("Fix applied! After: ")
print(df['active'].unique())

Fix applied! After: 
[ True False]


### 5. Date Parsing

In [24]:
print("Before: ")
print(df['start_date'].dtype)

Before: 
object


In [26]:
df['start_date'] = pd.to_datetime(df['start_date'], errors='coerce')
df['end_date'] = pd.to_datetime(df['end_date'], dayfirst=True, errors='coerce')

print("Fix applied! After: ")
print(df['start_date'].dtype)

Fix applied! After: 
datetime64[ns]


### 6. Logical Integrity, Clicks vs Impressions

###### Removing duplicate columns

In [30]:
df = df.loc[:, ~df.columns.duplicated()]
df

,campaign_id,campaign_name,start_date,end_date,channel,impressions,clicks,spend,conversions,active,campaign_tag
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24,2023-12-13,TikTok,16795,197,102.82,20.0,True,TI
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06,2023-05-12,Facebook,1860,30,24.33,1.0,False,FA
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13,2023-12-20,Email,77820,843,1323.39,51.0,False,EM
3,CMP-00004,Q1_BlackFriday_CMP-00004,NaT,2023-11-03,TikTok,55886,2019,2180.38,135.0,False,TI
4,CMP-00005,Q2_Winter_CMP-00005,2023-04-22,2023-04-23,Facebook,7265,169,252.44,30.0,True,FA
...,...,...,...,...,...,...,...,...,...,...,...
2015,CMP-00400,Q3_Summer_CMP-00400,2023-10-31,2023-11-13,TikTok,30592,586,503.95,77.0,True,TI
2016,CMP-01255,Q4_Summer_CMP-01255,2023-09-01,2023-09-26,Google Ads,20097,897,1641.00,162.0,False,GO
2017,CMP-01050,Q2_Launch_CMP-01050,2023-02-09,2023-02-21,Instagram,33254,1117,883.82,214.0,False,IN
2018,CMP-01118,Q4_Winter_CMP-01118,2023-03-30,2023-04-27,Facebook,68728,2960,4198.50,591.0,True,FA


In [31]:
not_possible = df['clicks'] > df['impressions']
print(df.loc[not_possible, ['campaign_id', 'impressions', 'clicks']].head(3))

Empty DataFrame
Columns: [campaign_id, impressions, clicks]
Index: []


### 7. logical integrity, Time

In [32]:
print("Before: ")
time_travel_mask = df['end_date'] < df['start_date']
print(df.loc[time_travel_mask, ['campaign_id', 'start_date', 'end_date']].head(3))

Before: 
   campaign_id start_date   end_date
23   CMP-00024 2023-05-06 2023-05-01
54   CMP-00055 2023-09-01 2023-08-27
71   CMP-00072 2023-02-01 2023-01-27


In [33]:
df.loc[time_travel_mask, 'end_date'] = df.loc[time_travel_mask, 'start_date'] + pd.Timedelta(days=30)
print("Fix applied! After: ")
print(df.loc[time_travel_mask, ['campaign_id', 'start_date', 'end_date']].head(3))

Fix applied! After: 
   campaign_id start_date   end_date
23   CMP-00024 2023-05-06 2023-06-05
54   CMP-00055 2023-09-01 2023-10-01
71   CMP-00072 2023-02-01 2023-03-03


### 8. Handling outliers

In [34]:
# Cap outliers in 'spend' column using IQR
Q1 = df['spend'].quantile(0.25)
Q3 = df['spend'].quantile(0.75)
IQR = Q3 - Q1
upper_limit = Q3 + (3 * IQR)

outlier_values = df['spend'] > upper_limit
print("Before:")
print(df.loc[outlier_values, ['campaign_id', 'spend']].head(3))

Before:
     campaign_id      spend
789    CMP-00790  500000.00
1443   CMP-01444    8921.51
1460   CMP-01461  500000.00


In [35]:
print("FIX APPLIED! After: ")
df.loc[outlier_values, 'spend'] = upper_limit
print(df.loc[outlier_values, ['campaign_id', 'spend']].head(3))

FIX APPLIED! After: 
     campaign_id      spend
789    CMP-00790  8603.5375
1443   CMP-01444  8603.5375
1460   CMP-01461  8603.5375


### 9. String parsing

In [36]:
print("Before:")
print(df['campaign_name'].head(3))

Before:
0    Q4_Summer_CMP-00001
1    Q1_Launch_CMP-00002
2    Q3_Winter_CMP-00003
Name: campaign_name, dtype: object


In [37]:
df['season'] = df['campaign_name'].str.extract(r'Q\d_([^_]+)_')
print("Fix applied! After:")
print(df[['campaign_name', 'season']].head(3))

Fix applied! After:
         campaign_name  season
0  Q4_Summer_CMP-00001  Summer
1  Q1_Launch_CMP-00002  Launch
2  Q3_Winter_CMP-00003  Winter


In [38]:
df

,campaign_id,campaign_name,start_date,end_date,channel,impressions,clicks,spend,conversions,active,campaign_tag,season
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24,2023-12-13,TikTok,16795,197,102.82,20.0,True,TI,Summer
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06,2023-05-12,Facebook,1860,30,24.33,1.0,False,FA,Launch
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13,2023-12-20,Email,77820,843,1323.39,51.0,False,EM,Winter
3,CMP-00004,Q1_BlackFriday_CMP-00004,NaT,2023-11-03,TikTok,55886,2019,2180.38,135.0,False,TI,BlackFriday
4,CMP-00005,Q2_Winter_CMP-00005,2023-04-22,2023-04-23,Facebook,7265,169,252.44,30.0,True,FA,Winter
...,...,...,...,...,...,...,...,...,...,...,...,...
2015,CMP-00400,Q3_Summer_CMP-00400,2023-10-31,2023-11-13,TikTok,30592,586,503.95,77.0,True,TI,Summer
2016,CMP-01255,Q4_Summer_CMP-01255,2023-09-01,2023-09-26,Google Ads,20097,897,1641.00,162.0,False,GO,Summer
2017,CMP-01050,Q2_Launch_CMP-01050,2023-02-09,2023-02-21,Instagram,33254,1117,883.82,214.0,False,IN,Launch
2018,CMP-01118,Q4_Winter_CMP-01118,2023-03-30,2023-04-27,Facebook,68728,2960,4198.50,591.0,True,FA,Winter
